# Algorithms and Computational Limits

```{contents}
:local:
:depth: 2
```

Finish the course by linking contracts, correctness, cost, and requirements. A large search space motivates better algorithms and narrower requirements, but does not itself prove that every exact algorithm must enumerate that space. Download complete implementations and checks: [AdvancedChecks.cs](../../demos/25/AdvancedChecks.cs).

## Growth Models Are Not Benchmarks

```{index} Growth Models Are Not Benchmarks
```

Use exact integer arithmetic to compare candidate work functions. A subset search may have 2^n candidate sets; positional orderings have n! candidates. Models describe possible work, not measured runtime. A budget comparison needs declared units.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

foreach(int n in new[]{10,20,30,40})
{
 BigInteger subsets=BigInteger.One<<n,factorial=BigInteger.One;
 for(int i=2;i<=n;i++)factorial*=i;
 Console.WriteLine($"n={n}: quadratic={(BigInteger)n*n}, subsets={subsets}, permutations={factorial}");
}
BigInteger budget=1_000_000,candidates=BigInteger.One<<30;
Console.WriteLine($"2^30 exceeds one-million-candidate budget={candidates>budget}");


Constants, hardware, data distribution, preprocessing, memory, and required output affect actual feasibility. Pruning can help without changing an exponential worst case. Dynamic programming may replace a choice tree with a state graph, yet its numeric-state range can still be large. Enumerating a large family is not a lower-bound proof for a decision or optimization problem.

## Exact Search and a Concrete Heuristic

```{index} Exact Search and a Concrete Heuristic
```

A manager selects the fewest teams covering six required skills. Team masks are A={0,1,2,3}, B={0,1,4}, C={2,3,5}. Greedily selecting the team covering the most still-uncovered skills picks A, then B and C: three teams. The feasible pair B,C covers all six with two.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

int[] teams={0b001111,0b010011,0b101100};
var greedy=TeamCover.Greedy(6,teams);var exact=TeamCover.Exact(6,teams)!;
Console.WriteLine($"greedy IDs={string.Join(",",greedy.Indices)}, count={greedy.Indices.Length}, coverage={greedy.Covered}");
Console.WriteLine($"exact IDs={string.Join(",",exact.Indices)}, count={exact.Indices.Length}, coverage={exact.Covered}");
public static class TeamCover
{
    public record Plan(int[] Indices,int Covered);
    // Teams are positions; each mask identifies covered requirements, up to 20.
    public static Plan Greedy(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);int covered=0;var chosen=new List<int>();
        while(covered!=full)
        {
            int best=-1,gain=0;
            for(int i=0;i<teams.Length;i++)
            {int next=System.Numerics.BitOperations.PopCount((uint)(teams[i]&~covered));if(next>gain){best=i;gain=next;}}
            if(best<0)break;
            chosen.Add(best);covered|=teams[best];
        }
        return new Plan(chosen.ToArray(),covered);
    }
    public static Plan? Exact(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);
        if(teams.Length>20)throw new ArgumentException("Exact teaching search is capped at 20 teams.");
        int bestCount=int.MaxValue,bestMask=0;
        for(int mask=0;mask<(1<<teams.Length);mask++)
        {
            int count=System.Numerics.BitOperations.PopCount((uint)mask);
            if(count>=bestCount)continue;
            int covered=0;for(int i=0;i<teams.Length;i++)if((mask&(1<<i))!=0)covered|=teams[i];
            if(covered==full){bestCount=count;bestMask=mask;}
        }
        return bestCount==int.MaxValue?null:new Plan(Enumerable.Range(0,teams.Length).Where(i=>(bestMask&(1<<i))!=0).ToArray(),full);
    }
    private static int Validate(int requirements,int[] teams)
    {
        ArgumentNullException.ThrowIfNull(teams);
        if(requirements<0||requirements>20)throw new ArgumentOutOfRangeException(nameof(requirements));
        int full=(1<<requirements)-1;
        if(teams.Any(mask=>mask<0||(mask&~full)!=0))throw new ArgumentException("Unknown requirement bit.");
        return full;
    }
}


The exact method enumerates all team subsets up to the teaching limit of 20 teams; it skips candidates whose counts cannot improve the current best. This is safe pruning because adding a candidate of at least the current best count cannot improve a minimum-count objective. The greedy method deterministically breaks equal gains by first position. For this example, no one team covers all requirements, so the feasible two-team witness is optimal independently of the enumeration output.

## Guarantees, Heuristics, and Unknown Answers

```{index} Guarantees, Heuristics, and Unknown Answers
```

An exact algorithm must return an optimum or a proven no-solution answer when it completes. A heuristic may return a useful feasible result without an optimality claim. An approximation guarantee is a proved bound relating the returned objective to the optimum under stated assumptions; good performance on samples is not such a proof.

The cover heuristic here is used as a verified greedy counterexample; this chapter does not claim or derive a general approximation ratio for it. A zero-gain stop means remaining requirements are uncovered by every available team. Budget exhaustion in a general search instead means unknown unless a separate proof establishes infeasibility. Report coverage separately from team count: a smaller incomplete selection is not a better feasible solution. Changing requirements, permitting partial coverage, or limiting input sizes changes the problem contract.

## Validate Results Before Comparing Speed

```{index} Validate Results Before Comparing Speed
```

For each strategy verify that returned positions are distinct and legal, that their union equals claimed coverage, and that full coverage is achieved before comparing counts. Test zero requirements, empty teams, and impossible requirements.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

foreach(var sample in new[]{(0,Array.Empty<int>()),(2,new[]{0b01}),(2,new[]{0b01,0b10})})
{
 var g=TeamCover.Greedy(sample.Item1,sample.Item2);var e=TeamCover.Exact(sample.Item1,sample.Item2);
 int union=g.Indices.Aggregate(0,(mask,i)=>mask|sample.Item2[i]);
 if(union!=g.Covered||g.Indices.Distinct().Count()!=g.Indices.Length)throw new Exception("Invalid coverage witness.");
 Console.WriteLine($"requirements={sample.Item1}, greedy full={g.Covered==((1<<sample.Item1)-1)}, exact count={e?.Indices.Length.ToString()??"unreachable"}");
}
public static class TeamCover
{
    public record Plan(int[] Indices,int Covered);
    // Teams are positions; each mask identifies covered requirements, up to 20.
    public static Plan Greedy(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);int covered=0;var chosen=new List<int>();
        while(covered!=full)
        {
            int best=-1,gain=0;
            for(int i=0;i<teams.Length;i++)
            {int next=System.Numerics.BitOperations.PopCount((uint)(teams[i]&~covered));if(next>gain){best=i;gain=next;}}
            if(best<0)break;
            chosen.Add(best);covered|=teams[best];
        }
        return new Plan(chosen.ToArray(),covered);
    }
    public static Plan? Exact(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);
        if(teams.Length>20)throw new ArgumentException("Exact teaching search is capped at 20 teams.");
        int bestCount=int.MaxValue,bestMask=0;
        for(int mask=0;mask<(1<<teams.Length);mask++)
        {
            int count=System.Numerics.BitOperations.PopCount((uint)mask);
            if(count>=bestCount)continue;
            int covered=0;for(int i=0;i<teams.Length;i++)if((mask&(1<<i))!=0)covered|=teams[i];
            if(covered==full){bestCount=count;bestMask=mask;}
        }
        return bestCount==int.MaxValue?null:new Plan(Enumerable.Range(0,teams.Length).Where(i=>(bestMask&(1<<i))!=0).ToArray(),full);
    }
    private static int Validate(int requirements,int[] teams)
    {
        ArgumentNullException.ThrowIfNull(teams);
        if(requirements<0||requirements>20)throw new ArgumentOutOfRangeException(nameof(requirements));
        int full=(1<<requirements)-1;
        if(teams.Any(mask=>mask<0||(mask&~full)!=0))throw new ArgumentException("Unknown requirement bit.");
        return full;
    }
}


For t teams, exhaustive enumeration costs O(t·2^t) time and O(t) auxiliary/output state in this implementation; greedy takes O(t·r) scans for r≤requirements selected teams, using fixed-width masks for at most 20 requirements. These are different algorithms with different quality contracts. Bit operations are constant time only under the fixed-width model used here.

## A Reviewable Final Comparison

```{index} A Reviewable Final Comparison
```

Use a problem from your semester project or this chapter’s service network. Specify objective, input model, legal inputs, tie policies, output, and resource budget. Compare at least two strategies on identical cases. Include independently checked witnesses, exact operation-count definitions, qualified time/space costs, and a recommendation tied to the requirements.

Suitable comparisons include BFS versus DFS for reachability (BFS adds minimum-hop information), direct versus prepared-pattern matching under the same ordinal/overlap policy, or exact versus greedy team coverage with feasibility and quality reported separately. Keep any durable VS Code/GitHub capstone submission in the existing project/appendix track; this content section supplies the comparison method, not a new required assignment. Include empty, disconnected, repeated-prefix, and infeasible cases as appropriate. A defensible conclusion states where both algorithms agree, what extra guarantee one offers, and where the budget or data model changes the recommendation.

## Exercise

```{index} Exercise
```

Validate the six-skill cover counterexample, establish why two is optimal, and reject an unknown requirement bit. Explain why the greedy result is feasible but not optimal.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

int[] teams={0b001111,0b010011,0b101100};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
int[] teams={0b001111,0b010011,0b101100};
var g=TeamCover.Greedy(6,teams);var e=TeamCover.Exact(6,teams)!;
if(g.Covered!=63||e.Covered!=63||e.Indices.Length!=2||teams.Any(mask=>mask==63))throw new Exception("Cover counterexample failed.");
Console.WriteLine($"feasible greedy={g.Indices.Length}, exact={e.Indices.Length}; no singleton covers all");
try{TeamCover.Greedy(2,new[]{0b100});}catch(ArgumentException){Console.WriteLine("unknown skill rejected");}
public static class TeamCover
{
    public record Plan(int[] Indices,int Covered);
    // Teams are positions; each mask identifies covered requirements, up to 20.
    public static Plan Greedy(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);int covered=0;var chosen=new List<int>();
        while(covered!=full)
        {
            int best=-1,gain=0;
            for(int i=0;i<teams.Length;i++)
            {int next=System.Numerics.BitOperations.PopCount((uint)(teams[i]&~covered));if(next>gain){best=i;gain=next;}}
            if(best<0)break;
            chosen.Add(best);covered|=teams[best];
        }
        return new Plan(chosen.ToArray(),covered);
    }
    public static Plan? Exact(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);
        if(teams.Length>20)throw new ArgumentException("Exact teaching search is capped at 20 teams.");
        int bestCount=int.MaxValue,bestMask=0;
        for(int mask=0;mask<(1<<teams.Length);mask++)
        {
            int count=System.Numerics.BitOperations.PopCount((uint)mask);
            if(count>=bestCount)continue;
            int covered=0;for(int i=0;i<teams.Length;i++)if((mask&(1<<i))!=0)covered|=teams[i];
            if(covered==full){bestCount=count;bestMask=mask;}
        }
        return bestCount==int.MaxValue?null:new Plan(Enumerable.Range(0,teams.Length).Where(i=>(bestMask&(1<<i))!=0).ToArray(),full);
    }
    private static int Validate(int requirements,int[] teams)
    {
        ArgumentNullException.ThrowIfNull(teams);
        if(requirements<0||requirements>20)throw new ArgumentOutOfRangeException(nameof(requirements));
        int full=(1<<requirements)-1;
        if(teams.Any(mask=>mask<0||(mask&~full)!=0))throw new ArgumentException("Unknown requirement bit.");
        return full;
    }
}


feasible greedy=3, exact=2; no singleton covers all
unknown skill rejected


```{rubric} Footnotes
```
[^1]: An exponential implementation does not prove an exponential lower bound for all algorithms solving its problem.
[^2]: This section distinguishes exactness, feasibility, empirical quality, and proved approximation bounds; it does not assert a complexity-class theorem.